In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [2]:
df=pd.read_csv("/content/heart_disease_data.csv")

In [3]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [4]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [5]:
df.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'],
      dtype='object')

In [6]:
df[['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal']]=df[['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal']].replace(0,np.nan)

In [7]:
df.isnull().sum()

,0
age,0
sex,96
cp,143
trestbps,0
chol,0
fbs,258
restecg,147
thalach,0
exang,204
oldpeak,99


In [8]:
def median_target(var):
  temp=df[df[var].notnull()]
  temp=temp[[var,'target']].groupby(['target'])[[var]].median().reset_index()
  return temp

In [9]:
columns=df.columns
columns=columns.drop('target')
for i in columns:
  median_target(i)
  df.loc[(df['target']==0) & (df[i].isnull()),i]=median_target(i)[i][0]
  df.loc[(df['target']==1) & (df[i].isnull()),i]=median_target(i)[i][1]

In [10]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [11]:
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,303.000000,303.0,303.000000,303.000000,303.000000,303.0,303.000000,303.000000,303.0,303.000000,303.000000,303.000000,303.000000,303.000000
mean,54.366337,1.0,1.910891,131.623762,246.264026,1.0,1.013201,149.646865,1.0,1.407921,1.498350,1.455446,2.330033,0.544554
std,9.082101,0.0,0.483482,17.538143,51.830751,0.0,0.114325,22.905161,0.0,0.954114,0.500824,0.693271,0.583994,0.498835
min,29.000000,1.0,1.000000,94.000000,126.000000,1.0,1.000000,71.000000,1.0,0.100000,1.000000,1.000000,1.000000,0.000000
25%,47.500000,1.0,2.000000,120.000000,211.000000,1.0,1.000000,133.500000,1.0,0.900000,1.000000,1.000000,2.000000,0.000000
50%,55.000000,1.0,2.000000,130.000000,240.000000,1.0,1.000000,153.000000,1.0,1.000000,1.000000,1.000000,2.000000,1.000000
75%,61.000000,1.0,2.000000,140.000000,274.500000,1.0,1.000000,166.000000,1.0,1.800000,2.000000,2.000000,3.000000,1.000000
max,77.000000,1.0,3.000000,200.000000,564.000000,1.0,2.000000,202.000000,1.0,6.200000,2.000000,4.000000,3.000000,1.000000


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
 13  target    303 non-null    int64  
dtypes: float64(9), int64(5)
memory usage: 33.3 KB


In [13]:
df['target'].value_counts()

,count
target,
1,165
0,138


In [14]:
for feature in df:
  Q1=df[feature].quantile(0.25)
  Q3=df[feature].quantile(0.75)
  IQR=Q3-Q1
  lower=Q1-1.5*IQR
  upper=Q3+1.5*IQR
  df[feature]=np.where(df[feature]>upper,upper,np.where(df[feature]<lower,lower,df[feature]))

1->Defective hearth

0-->Healthy hearth

In [15]:
X=df.drop(columns='target',axis=1)
y=df['target']

In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=2)

In [17]:
print(X.shape,X_train.shape,X_test.shape)


(303, 13) (242, 13) (61, 13)


In [18]:
from xgboost import XGBClassifier
model=XGBClassifier()
model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [19]:
y_pred=model.predict(X_test)

In [20]:
accuracy_score(y_test,y_pred)

0.8852459016393442

In [21]:
import pickle
with open("heart.pkl",'wb') as file:
  pickle.dump(model,file)

In [22]:
X_train.columns

Index(['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach',
       'exang', 'oldpeak', 'slope', 'ca', 'thal'],
      dtype='object')